# 12c - Modality dispatch + city-agnostic ensemble (toward a Ukraine-wide model)

> Copyright (C) 2024-2026 Marco Heinzen - SPDX-License-Identifier: AGPL-3.0-or-later
> Part of the Master Thesis "Building Damage Assessment with Multimodal Satellite Time Series and Machine Learning in the Russia-Ukraine War 2022-2026"
> Code hosted at https://github.com/marcoheinzen/bda
> Parts of this code were written or improved with the assistance of Claude (Anthropic); all other code, and the concept, research, architecture, design, execution, testing and validation throughout, are the author's work.


Two questions, answered from pretrained-model OOF (run 12a first):
- **(A) Best model per modality combination** - route each city by its available sensors (has_card/has_coh/has_ms),
  evaluated **nested leave-one-city-out** (choose the model from OTHER cities, score on the held-out city). No
  per-city peeking = no leakage.
- **(B) City-agnostic ensemble** - average pretrained models' OOF probabilities (one global rule for all cities) and
  test whether it **tightens the per-city AUC spread** without losing the mean.

Honest framing: leave-one-city-out mean-folds = the expected performance on **unseen** Ukrainian cities. Run in WSL.

In [1]:
import sys, re
from pathlib import Path
import numpy as np
import json
import pandas as pd
import pyarrow.parquet as pq
from sklearn.metrics import roc_auc_score

_known = [Path("/content/drive_f/masterthesis/notebooks"),
          Path("/mnt/f/PROJECTS/masterthesis/gdrive/masterthesis/notebooks"),
          Path(r"F:\PROJECTS\masterthesis\gdrive\masterthesis\notebooks")]
_nb = next((c for c in list(Path.cwd().parents) + _known if (c / "global_setup.py").exists()), None)
if _nb is None: raise RuntimeError("global_setup.py not found")
if str(_nb) not in sys.path: sys.path.insert(0, str(_nb))
import global_setup as gs

DRIVE_ROOT = Path(gs.DRIVE_ROOT); RESULTS_ROOT = Path(gs.RESULTS_ROOT)
NB12 = RESULTS_ROOT / "nb12"
mat = pd.read_csv(NB12 / "nb12a_per_city_auc_matrix.csv", index_col=0)
summ = pd.read_csv(NB12 / "nb12a_meanfolds_vs_pooled.csv")

gk_path = Path(getattr(gs, "GROUPKFOLD_PATH", Path(gs.STACK_ROOT)/"groupkfold_assignments.json"))
assign = json.loads(Path(gk_path).read_text())
ca = assign["city_assignments"]
def combo_str(a):
    return "+".join([n for n,k in [("card","has_card"),("coh","has_coh"),("ms","has_ms")] if a.get(k)]) or "none"
city_combo = {c: combo_str(a) for c, a in ca.items()}
print("cities:", len(city_combo))
print(pd.Series(city_combo).value_counts().to_string())

def safe_auc(y, s):
    y=np.asarray(y); s=np.asarray(s)
    return float(roc_auc_score(y,s)) if len(np.unique(y))>1 else np.nan

/home/alpineobotics/miniconda3/envs/bda/lib/python3.12/site-packages/pyproj/network.py:59: UserWarning: pyproj unable to set PROJ database path.
  _set_context_ca_bundle_path(ca_bundle_path)


BDA GLOBAL SETUP
Started: 2026-06-28 00:51:38
Python: 3.12.12

[1/7] Directory Structure
----------------------------------------------------------------------
  GDrive (G:):       /content/drive_f/masterthesis OK
  GDrive (F:):       /content/drive_f/masterthesis OK
  Local data (G:):   /content/masterthesis_local/data OK
  Data stack (F:):   /mnt/f/PROJECTS/masterthesis/data_stack OK

  TIER_SELECTION: [0, 1, 2]
  CITY_SELECTION: None (tier filter)
  REQUIRE_UNOSAT: False
  CITIES_TO_PROCESS: 21 cities

[2/7] Credentials
----------------------------------------------------------------------
  Copernicus: inf***
  OpenTopography: OK
  Earthdata: marcoheinzen

[3/7] Python Packages
----------------------------------------------------------------------


/content/drive_f/masterthesis/notebooks/global_setup.py:564: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources



  Already installed: 23
  Newly installed:   0
  Failed:            0

[4/7] Global Imports & Configuration
----------------------------------------------------------------------
  All imports loaded

[5/7] Processing Config & SNAP
----------------------------------------------------------------------
  GPT: Usage:
  Temporal baseline: 10-24 days
  Wavelength: 0.0555

[6/7] GPU Status
----------------------------------------------------------------------
  CUDA available: NVIDIA GeForce RTX 2070 SUPER
    CUDA version: 12.8

[7/7] Disk Space
----------------------------------------------------------------------
  GDrive (G:)     937.8/7452.0 GB (6514.3 GB free)
  GDrive (F:)     1405.8/3726.0 GB (2320.2 GB free)
  Local data      11566.2/14901.9 GB (3335.6 GB free)
  Data stack      1405.8/3726.0 GB (2320.2 GB free)
  WSL ext4        69.0/1006.9 GB (886.6 GB free)

GLOBAL SETUP COMPLETE
  Torch device: cuda
  Cities: 21, CITY=Avdiivka
  Functions: load_aoi(), load_aoi_gdf(), load_aoi_

## Candidate experiments

`CANDIDATES` is the pool the dispatcher and ensemble draw from. Default = experiments scored on enough cities;
restrict it to your reported set if you prefer (matrix index names).

In [2]:
MIN_CITIES = 12
cand_df = summ[summ["n_cities_scored"] >= MIN_CITIES].sort_values("mean_folds_auc", ascending=False)
CANDIDATES = [e for e in cand_df["experiment"].tolist() if e in mat.index]
print(f"{len(CANDIDATES)} candidate experiments (>= {MIN_CITIES} cities scored). Top 10:")
print(cand_df[["experiment","mean_folds_auc","n_cities_scored"]].head(10).to_string(index=False))

396 candidate experiments (>= 12 cities scored). Top 10:
                          experiment  mean_folds_auc  n_cities_scored
               NB08b_BDA_RGB_XGBoost        0.748478               21
              NB08b_BDA_RGB_AdaBoost        0.748367               21
NB08b_BDA_COH_DROP+RGB+CARD_AdaBoost        0.745562               12
                NB08b_BDA_RGB_LogReg        0.744569               21
     NB08b_BDA_COH_DROP+RGB_AdaBoost        0.744086               12
            NB08b_BDA_RGB_ExtraTrees        0.743519               21
                   NB08b_BDA_RGB_GBM        0.743394               21
          NB08b_BDA_RGB_UISEM_voting        0.742090               21
                NB08b_BDA_RGB_RF-200        0.741769               21
                          CLF_F7_GBM        0.741301               19


## (A) Modality dispatch - nested leave-one-city-out

For each held-out city h (combo k): candidate models that scored h; pick the one with the best mean AUC over the
OTHER cities sharing combo k (fallback: all other cities if <2 same-combo). Dispatched AUC(h) = chosen model's AUC on
h. Compare to the single best overall model on the same cities.

In [3]:
cities = [c for c in mat.columns if c in city_combo]
disp_auc, choice = {}, {}
for h in cities:
    k = city_combo[h]
    cand = [e for e in CANDIDATES if e in mat.index and not pd.isna(mat.loc[e, h])]
    if not cand: continue
    same = [c for c in cities if c != h and city_combo[c] == k]
    pool = same if len(same) >= 2 else [c for c in cities if c != h]
    best_e, best_v = None, -np.inf
    for e in cand:
        vals = [mat.loc[e, c] for c in pool if not pd.isna(mat.loc[e, c])]
        if vals:
            mv = float(np.mean(vals))
            if mv > best_v: best_v, best_e = mv, e
    if best_e is not None:
        disp_auc[h] = float(mat.loc[best_e, h]); choice[h] = (k, best_e)

disp_cities = list(disp_auc)
disp_mean = float(np.mean([disp_auc[c] for c in disp_cities])) if disp_cities else np.nan
disp_std  = float(np.std([disp_auc[c] for c in disp_cities])) if disp_cities else np.nan

single_means = {e: float(np.nanmean([mat.loc[e, c] for c in disp_cities])) for e in CANDIDATES}
best_single = max(single_means, key=single_means.get)
print(f"cities dispatched: {len(disp_cities)}")
print(f"DISPATCH  mean-folds = {disp_mean:.4f} +/- {disp_std:.4f}")
print(f"BEST SINGLE ({best_single[:40]}) on same cities = {single_means[best_single]:.4f}")
print(f"delta (dispatch - best single) = {disp_mean - single_means[best_single]:+.4f}")

print("\nlearned dispatch table (combo -> chosen model, mode):")
dt = {}
for h,(k,e) in choice.items(): dt.setdefault(k, []).append(e)
for k, es in sorted(dt.items()):
    top = pd.Series(es).value_counts().idxmax()
    print(f"  {k:14s} -> {top}   (cities: {len(es)})")
pd.DataFrame([{"city":c,"combo":choice[c][0],"chosen":choice[c][1],"auc":disp_auc[c]} for c in disp_cities]
            ).to_csv(NB12/"nb12c_dispatch_per_city.csv", index=False)
print("saved:", NB12/"nb12c_dispatch_per_city.csv")

cities dispatched: 21
DISPATCH  mean-folds = 0.7367 +/- 0.0821
BEST SINGLE (NB08b_BDA_RGB_XGBoost) on same cities = 0.7485
delta (dispatch - best single) = -0.0118

learned dispatch table (combo -> chosen model, mode):
  card+coh       -> NB08b_BDA_RGB_XGBoost   (cities: 2)
  card+coh+ms    -> NB08b_BDA_RGB_AdaBoost   (cities: 16)
  card+ms        -> NB08b_BDA_RGB_LogReg   (cities: 3)
saved: /content/drive_f/masterthesis/results/nb12/nb12c_dispatch_per_city.csv


## (B) City-agnostic ensemble (global soft-voting)

Average the held-out probabilities of a set of building-level (V2) models, joined on `building_id` - one global rule
for all cities. Compares the ensemble's per-city spread and mean to its members on the SAME shared buildings.

In [4]:
ENSEMBLE = CANDIDATES[:4]      # set explicitly to your members if preferred
print("ensemble members:")
for e in ENSEMBLE: print("  ", e)

# map experiment sig -> latest OOF path (both roots)
roots = [Path(getattr(gs,a)) for a in ["RESULTS_ROOT","OUTPUT_ROOT","DATA_OUTPUTS"] if getattr(gs,a,None)]
roots += [DRIVE_ROOT/"data"/"outputs", RESULTS_ROOT]
seen=set(); roots=[r for r in roots if r.exists() and (r not in seen and not seen.add(r))]
def sig_of(n):
    s=re.sub(r"\.parquet$","",n); s=re.sub(r"^oof_","",s)
    return re.sub(r"__\d{8}_\d{6}_[0-9a-fA-F]+$","",s)
bysig={}
for r in roots:
    for p in r.rglob("oof_*.parquet"):
        m=re.search(r"(\d{8}_\d{6})", p.name)
        bysig.setdefault(sig_of(p.name), []).append((m.group(1) if m else "", p))
def latest(sig): return sorted(bysig.get(sig, []))[-1][1] if sig in bysig else None

frames=[]; ok=[]
for e in ENSEMBLE:
    p=latest(e)
    if p is None: print("  no OOF for", e); continue
    cc=[f.name for f in pq.ParquetFile(p).schema_arrow]
    if "building_id" not in cc:
        print("  skip (not building-level):", e); continue
    use=[c for c in ["building_id","city","y_true","y_proba","is_final"] if c in cc]
    d=pd.read_parquet(p, columns=use)
    if "is_final" in d: d=d[~d["is_final"].astype(bool)]
    d=d[["building_id","city","y_true","y_proba"]].rename(columns={"y_proba":f"p__{e[:24]}"})
    frames.append(d.set_index(["building_id","city","y_true"])); ok.append(e)

if len(frames) >= 2:
    J = pd.concat(frames, axis=1, join="inner").reset_index()
    pcols = [c for c in J.columns if c.startswith("p__")]
    J["p_ensemble"] = J[pcols].mean(axis=1)
    print(f"\nshared buildings across {len(ok)} members: {len(J):,}")

    def per_city(col):
        return {c: safe_auc(g["y_true"], g[col]) for c,g in J.groupby("city")}
    def summ_(d):
        v=np.array([x for x in d.values() if x==x]); return (float(np.mean(v)), float(np.std(v)), len(v))
    rows=[]
    for col in pcols+["p_ensemble"]:
        m,s,n=summ_(per_city(col))
        pooled=safe_auc(J["y_true"], J[col])
        rows.append({"model":col,"mean_folds":m,"std_folds":s,"n_cities":n,"pooled":pooled})
    comp=pd.DataFrame(rows).sort_values("mean_folds", ascending=False)
    comp.to_csv(NB12/"nb12c_ensemble_vs_members.csv", index=False)
    print("\n(on shared buildings) ensemble vs members:\n")
    print(comp.to_string(index=False))
    ens=comp[comp["model"]=="p_ensemble"].iloc[0]
    best_mem=comp[comp["model"]!="p_ensemble"].iloc[0]
    print(f"\nensemble mean-folds {ens['mean_folds']:.4f} (std {ens['std_folds']:.4f}) vs "
          f"best member {best_mem['mean_folds']:.4f} (std {best_mem['std_folds']:.4f})")
    print(f"spread change (ensemble std - best member std): {ens['std_folds']-best_mem['std_folds']:+.4f}")
    print("saved:", NB12/"nb12c_ensemble_vs_members.csv")
else:
    print("need >=2 building-level members for the ensemble")

ensemble members:
   NB08b_BDA_RGB_XGBoost
   NB08b_BDA_RGB_AdaBoost
   NB08b_BDA_COH_DROP+RGB+CARD_AdaBoost
   NB08b_BDA_RGB_LogReg

shared buildings across 4 members: 205,942

(on shared buildings) ensemble vs members:

                      model  mean_folds  std_folds  n_cities   pooled
p__NB08b_BDA_COH_DROP+RGB+C    0.745562   0.071589        12 0.738145
  p__NB08b_BDA_RGB_AdaBoost    0.743891   0.074934        12 0.739969
   p__NB08b_BDA_RGB_XGBoost    0.743807   0.073685        12 0.737544
                 p_ensemble    0.737154   0.075929        12 0.744471
    p__NB08b_BDA_RGB_LogReg    0.734402   0.076259        12 0.748535

ensemble mean-folds 0.7372 (std 0.0759) vs best member 0.7456 (std 0.0716)
spread change (ensemble std - best member std): +0.0043
saved: /content/drive_f/masterthesis/results/nb12/nb12c_ensemble_vs_members.csv


## Interpretation (for the thesis)

- If **dispatch delta > 0**, modality-aware routing beats any single fixed model - report the dispatch table as the
  recommended deployment rule, with the nested mean-folds as its honest score.
- If the **ensemble std < best member std** with similar mean, combination buys **stability across cities** (a
  city-agnostic gain) even when it does not raise the mean - consistent with "classifier/ensemble choice is
  second-order" but valuable for Ukraine-wide deployment.
- Either way, the **leave-one-city-out mean-folds is the expected unseen-city performance**; the per-city spread
  (0.38-0.65 range) is why the product is a screening layer (cue VHR), not a per-building verdict.
- Leakage guard honoured: dispatch key = a-priori modality availability; ensemble = one global rule. No per-city
  selection or weighting using held-out labels.